# Задание

1. решить задачу многоклассовой класификации любыми доступными способами
2. таргет - это 7 дефектов (последние 7 столбцов)


# Установка библиотек

In [1]:
# %pip install pandas numpy plotly nbformat scikit-learn xgboost

# Импорты

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import abc as abc
import sklearn as skl

# Дата сет

## Загружаем

In [3]:
df = pd.read_csv('steelplate.csv')


df_clean = df.copy()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19219 entries, 0 to 19218
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     19219 non-null  int64  
 1   X_Minimum              19219 non-null  int64  
 2   X_Maximum              19219 non-null  int64  
 3   Y_Minimum              19219 non-null  int64  
 4   Y_Maximum              19219 non-null  int64  
 5   Pixels_Areas           19219 non-null  int64  
 6   X_Perimeter            19219 non-null  int64  
 7   Y_Perimeter            19219 non-null  int64  
 8   Sum_of_Luminosity      19219 non-null  int64  
 9   Minimum_of_Luminosity  19219 non-null  int64  
 10  Maximum_of_Luminosity  19219 non-null  int64  
 11  Length_of_Conveyer     19219 non-null  int64  
 12  TypeOfSteel_A300       19219 non-null  int64  
 13  TypeOfSteel_A400       19219 non-null  int64  
 14  Steel_Plate_Thickness  19219 non-null  int64  
 15  Ed

In [4]:
df.describe()

,id,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,...,Orientation_Index,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults
count,19219.000000,19219.000000,19219.000000,1.921900e+04,1.921900e+04,19219.000000,19219.000000,19219.000000,1.921900e+04,19219.000000,...,19219.000000,19219.000000,19219.000000,19219.000000,19219.000000,19219.000000,19219.000000,19219.000000,19219.000000,19219.000000
mean,9609.000000,709.854675,753.857641,1.849756e+06,1.846605e+06,1683.987616,95.654665,64.124096,1.918467e+05,84.808419,...,0.102742,-0.138382,0.571902,0.076279,0.059837,0.178573,0.029554,0.025235,0.247828,0.341225
std,5548.191747,531.544189,499.836603,1.903554e+06,1.896295e+06,3730.319865,177.821382,101.054178,4.420247e+05,28.800344,...,0.487681,0.120344,0.332219,0.265450,0.237190,0.383005,0.169358,0.156844,0.431762,0.474133
min,0.000000,0.000000,4.000000,6.712000e+03,6.724000e+03,6.000000,2.000000,1.000000,2.500000e+02,0.000000,...,-0.988400,-0.885000,0.119000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4804.500000,49.000000,214.000000,6.574680e+05,6.575020e+05,89.000000,15.000000,14.000000,9.848000e+03,70.000000,...,-0.272700,-0.192500,0.253200,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,9609.000000,777.000000,796.000000,1.398169e+06,1.398179e+06,168.000000,25.000000,23.000000,1.823800e+04,90.000000,...,0.111100,-0.142600,0.472900,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,14413.500000,1152.000000,1165.000000,2.368032e+06,2.362511e+06,653.000000,64.000000,61.000000,6.797800e+04,105.000000,...,0.529400,-0.084000,0.999400,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,19218.000000,1705.000000,1713.000000,1.298766e+07,1.298769e+07,152655.000000,7553.000000,903.000000,1.159141e+07,196.000000,...,0.991700,0.642100,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [5]:
df.head()

,id,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,...,Orientation_Index,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults
0,0,584,590,909972,909977,16,8,5,2274,113,...,-0.5000,-0.0104,0.1417,0,0,0,1,0,0,0
1,1,808,816,728350,728372,433,20,54,44478,70,...,0.7419,-0.2997,0.9491,0,0,0,0,0,0,1
2,2,39,192,2212076,2212144,11388,705,420,1311391,29,...,-0.0105,-0.0944,1.0000,0,0,1,0,0,0,0
3,3,781,789,3353146,3353173,210,16,29,3202,114,...,0.6667,-0.0402,0.4025,0,0,1,0,0,0,0
4,4,1540,1560,618457,618502,521,72,67,48231,82,...,0.9158,-0.2455,0.9998,0,0,0,0,0,0,1


## Обработка данных

In [4]:
# Разделяем признаки и таргеты (последние 7 столбцов - это дефекты)
X = df_clean.iloc[:, :-7]  # Все столбцы кроме последних 7
y = df_clean.iloc[:, -7:]  # Последние 7 столбцов (таргеты)

print("Признаки (X):")
print(X.shape)
print("\nТаргеты (y):")
print(y.shape)
print("\nНазвания таргетов (дефектов):")
print(y.columns.tolist())

Признаки (X):
(19219, 28)

Таргеты (y):
(19219, 7)

Названия таргетов (дефектов):
['Pastry', 'Z_Scratch', 'K_Scatch', 'Stains', 'Dirtiness', 'Bumps', 'Other_Faults']


In [7]:
# Удалим столбец ID (если он есть)
if 'id' in X.columns:
    X = X.drop('id', axis=1)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, hamming_loss
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Масштабирование признаков
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Разделяем данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")
print(f"Таргеты - количество образцов с каждым дефектом:")
print(y.sum())

Размер обучающей выборки: (15375, 27)
Размер тестовой выборки: (3844, 27)
Таргеты - количество образцов с каждым дефектом:
Pastry          1466
Z_Scratch       1150
K_Scatch        3432
Stains           568
Dirtiness        485
Bumps           4763
Other_Faults    6558
dtype: int64


## Модель 1: Логистическая регрессия (MultiOutput)

In [ ]:
# Обучаем модель логистической регрессии
lr_model = MultiOutputClassifier(LogisticRegression(max_iter=1000, random_state=42))
lr_model.fit(X_train, y_train)

# Предсказываем
y_pred_lr = lr_model.predict(X_test)

# Оцениваем
print("=== Логистическая регрессия ===")
print(f"Hamming Loss: {hamming_loss(y_test, y_pred_lr):.4f}")
print(f"Точность (Accuracy): {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"F1-score (средний): {f1_score(y_test, y_pred_lr, average='weighted', zero_division=0):.4f}")

## Модель 2: Random Forest (MultiOutput)

In [ ]:
# Обучаем модель Random Forest
rf_model = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
rf_model.fit(X_train, y_train)

# Предсказываем
y_pred_rf = rf_model.predict(X_test)

# Оцениваем
print("=== Random Forest ===")
print(f"Hamming Loss: {hamming_loss(y_test, y_pred_rf):.4f}")
print(f"Точность (Accuracy): {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"F1-score (средний): {f1_score(y_test, y_pred_rf, average='weighted', zero_division=0):.4f}")

## Модель 3: Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Обучаем модель Gradient Boosting
gb_model = MultiOutputClassifier(GradientBoostingClassifier(n_estimators=100, random_state=42))
gb_model.fit(X_train, y_train)

# Предсказываем
y_pred_gb = gb_model.predict(X_test)

# Оцениваем
print("=== Gradient Boosting ===")
print(f"Hamming Loss: {hamming_loss(y_test, y_pred_gb):.4f}")
print(f"Точность (Accuracy): {accuracy_score(y_test, y_pred_gb):.4f}")
print(f"F1-score (средний): {f1_score(y_test, y_pred_gb, average='weighted', zero_division=0):.4f}")

## Сравнение моделей

In [ ]:
# Создаем итоговую таблицу сравнения
results = {
    'Модель': ['Логистическая регрессия', 'Random Forest', 'XGBoost'],
    'Hamming Loss': [
        hamming_loss(y_test, y_pred_lr),
        hamming_loss(y_test, y_pred_rf),
        hamming_loss(y_test, y_pred_xgb)
    ],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    'F1-score (weighted)': [
        f1_score(y_test, y_pred_lr, average='weighted', zero_division=0),
        f1_score(y_test, y_pred_rf, average='weighted', zero_division=0),
        f1_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
    ]
}

results_df = pd.DataFrame(results)
print("\n=== ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ ===")
print(results_df.to_string(index=False))
print(f"\nЛучшая модель по Accuracy: {results_df.loc[results_df['Accuracy'].idxmax(), 'Модель']}")
print(f"Лучшая модель по F1-score: {results_df.loc[results_df['F1-score (weighted)'].idxmax(), 'Модель']}")

## Визуализация результатов

In [ ]:
# График сравнения моделей по Accuracy
fig = px.bar(results_df, x='Модель', y=['Accuracy', 'F1-score (weighted)'], 
             barmode='group', title='Сравнение моделей по метрикам')
fig.show()

# График Hamming Loss
fig2 = px.bar(results_df, x='Модель', y='Hamming Loss', 
              title='Hamming Loss для каждой модели (меньше - лучше)')
fig2.show()